# Tahap 03 — Text Chunking untuk Manual Coding dan BERTopic

## Judul Project

**Analisis Komputasional Topik Pidato Presiden Prabowo pada Forum Nasional dan Internasional Menggunakan Manual Coding dan BERTopic**

## Tujuan Notebook

Notebook ini digunakan untuk membagi naskah pidato hasil preprocessing menjadi unit analisis yang lebih kecil atau *chunk*.

Tahap ini penting karena:

1. Manual Coding membutuhkan unit teks yang cukup ringkas agar mudah diberi kode secara kualitatif.
2. BERTopic membutuhkan jumlah dokumen yang lebih banyak daripada hanya 6 pidato utuh.
3. Chunking membantu menjaga konteks teks tetap terbaca tanpa memotong kalimat secara sembarangan.
4. Output chunk akan menjadi input untuk tahap Manual Coding dan BERTopic.

## Input Utama

```text
data/processed/speech_preprocessed_master.csv
```

## Output Utama

```text
data/processed/speech_chunks_master.csv
data/processed/speech_chunks_for_bertopic.csv
reports/tables/chunking_summary_by_speech.csv
reports/tables/chunking_quality_report.csv
reports/tables/stage03_output_manifest.json
```

## 1. Import Library

Notebook ini hanya menggunakan library standar Python dan Pandas agar lebih mudah direplikasi di Jupyter Notebook lokal maupun Google Colab.

In [1]:
# ============================================================
# Import Library
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import sys

import pandas as pd

print("Library berhasil di-import.")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")

Library berhasil di-import.
Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.2.2


## 2. Setup Path Project

Cell ini mendeteksi root folder project secara otomatis.

Strategi deteksi:

1. Mulai dari current working directory.
2. Mencari folder yang memiliki `data/processed/speech_preprocessed_master.csv`.
3. Jika tidak ditemukan, fallback ke current working directory.
4. Membuat folder output jika belum tersedia.

In [2]:
# ============================================================
# Setup Path Project
# ============================================================

def find_project_root(start_path: Path | None = None) -> Path:
    """
    Mendeteksi root folder project berdasarkan keberadaan file input Tahap 02.
    """
    if start_path is None:
        start_path = Path.cwd().resolve()
    else:
        start_path = Path(start_path).resolve()

    candidate_paths = [start_path] + list(start_path.parents)

    for candidate in candidate_paths:
        expected_input = candidate / "data" / "processed" / "speech_preprocessed_master.csv"
        if expected_input.exists():
            return candidate

    return start_path


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORT_TABLE_DIR = REPORTS_DIR / "tables"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SPEECH_PREPROCESSED = PROCESSED_DIR / "speech_preprocessed_master.csv"

print("Project path berhasil disiapkan.")
print(f"Current working directory : {Path.cwd().resolve()}")
print(f"PROJECT_ROOT              : {PROJECT_ROOT}")
print(f"PROCESSED_DIR             : {PROCESSED_DIR}")
print(f"REPORT_TABLE_DIR          : {REPORT_TABLE_DIR}")
print(f"INPUT_SPEECH_PREPROCESSED : {INPUT_SPEECH_PREPROCESSED}")

Project path berhasil disiapkan.
Current working directory : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\notebooks
PROJECT_ROOT              : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech
PROCESSED_DIR             : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed
REPORT_TABLE_DIR          : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables
INPUT_SPEECH_PREPROCESSED : D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_preprocessed_master.csv


## 3. Preflight Check Input Tahap 02

Sebelum melakukan chunking, notebook akan memastikan file input dari Tahap 02 tersedia.

Jika file belum ditemukan, pastikan file berikut sudah ada:

```text
data/processed/speech_preprocessed_master.csv
```

In [3]:
# ============================================================
# Preflight Check Input
# ============================================================

if not INPUT_SPEECH_PREPROCESSED.exists():
    raise FileNotFoundError(
        f"File input Tahap 02 tidak ditemukan: {INPUT_SPEECH_PREPROCESSED}\n"
        "Pastikan Tahap 02 sudah dijalankan dan file speech_preprocessed_master.csv "
        "tersimpan di data/processed/."
    )

print("File input Tahap 02 ditemukan.")
print(f"Ukuran file: {INPUT_SPEECH_PREPROCESSED.stat().st_size:,} bytes")

File input Tahap 02 ditemukan.
Ukuran file: 666,480 bytes


## 4. Membaca Dataset Hasil Preprocessing

Dataset `speech_preprocessed_master.csv` dibaca sebagai input utama.

Notebook akan memeriksa struktur kolom secara eksplisit agar tidak ada asumsi kolom yang tidak valid.

In [4]:
# ============================================================
# Load Dataset Hasil Preprocessing
# ============================================================

speech_df = pd.read_csv(INPUT_SPEECH_PREPROCESSED)

print("Dataset berhasil dibaca.")
print(f"Shape dataset: {speech_df.shape}")

display(speech_df.head())
print("Daftar kolom:")
for col in speech_df.columns:
    print(f"- {col}")

Dataset berhasil dibaca.
Shape dataset: (6, 32)


,speech_id,file_name,text_body,file_size_bytes,encoding_used,char_count_body,word_count_body,line_count_body,content_sha256,speech_title_from_filename,...,text_preprocessed_tokens,text_for_manual_coding,text_for_bertopic,char_count_clean,word_count_clean,token_count_basic,unique_token_count_basic,text_clean_sha256,preprocessing_config_json,processed_stage02_at
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Distinguished Leaders of BRICS.\n\nIt is indee...,1888,utf-8,1642,256,8,a2f66ccd4a1d6ba1b4793089989eb09607add274f28050...,Brics Leaders,...,leaders brics indeed great honor join importan...,Distinguished Leaders of BRICS.\n\nIt is indee...,Distinguished Leaders of BRICS.\n\nIt is indee...,1642,254,138,101,8cd52135a957c95201d1430d1730dba226a5309a51ede8...,"{""remove_urls"": true, ""remove_email"": true, ""r...",2026-06-11T21:38:26
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Bismillahirrahmanirrahim.\n\nAssalamu’alaikum ...,24596,utf-8,24220,3417,82,2fe161589989ad5494b22841bb0c3bc26977e64fecbd0c...,Panen Raya,...,assalamu'alaikum selamat siang sejahtera perta...,Bismillahirrahmanirrahim.\n\nAssalamu'alaikum ...,Bismillahirrahmanirrahim.\n\nAssalamu'alaikum ...,24220,3388,1701,809,26334373815c3bc7667d520e4bb95f1a94aaa7a2620107...,"{""remove_urls"": true, ""remove_email"": true, ""r...",2026-06-11T21:38:26
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-...,Bismillahirrahmanirrahim.\n\nAssalamu’alaikum ...,18090,utf-8,17990,2543,54,881ebf65c5148e942e7fb142ee7950d6d91a5381cdbb46...,Peresmian Infrastruktur Energi,...,assalamu'alaikum selamat sore sejahtera energi...,Bismillahirrahmanirrahim.\n\nAssalamu'alaikum ...,Bismillahirrahmanirrahim.\n\nAssalamu'alaikum ...,17992,2507,1349,708,21a4c17de04d641e88f06fed801890b44f6bc962c4aba8...,"{""remove_urls"": true, ""remove_email"": true, ""r...",2026-06-11T21:38:26
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,"Bismillahirrahmanirrahim,\n\nAssalamu’alaikum ...",24811,utf-8,24459,3484,60,1452ea654813e9675ec6c863ef5f68f47d796cdbf5a01f...,Peresmian 166 Sekolah,...,assalamu'alaikum selamat pagi selamat siang se...,"Bismillahirrahmanirrahim,\n\nAssalamu'alaikum ...","Bismillahirrahmanirrahim,\n\nAssalamu'alaikum ...",24461,3441,1715,872,d4e324ae3e214de594d66380477793f6fe3b6582a51bff...,"{""remove_urls"": true, ""remove_email"": true, ""r...",2026-06-11T21:38:26
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,"Distinguished President and CEO of the WEF, Mr...",21485,utf-8,21135,3585,51,5c443ea53ec06c51d42ec156fbe3d3b5aa39f0917b65e2...,World Economic Forum,...,president ceo wef borge brende assalamu'alaiku...,"Distinguished President and CEO of the WEF, Mr...","Distinguished President and CEO of the WEF, Mr...",21135,3495,1728,854,64a1d89014f09ba15886bc525a4ac3bf55002778788602...,"{""remove_urls"": true, ""remove_email"": true, ""r...",2026-06-11T21:38:26


Daftar kolom:
- speech_id
- file_name
- text_body
- file_size_bytes
- encoding_used
- char_count_body
- word_count_body
- line_count_body
- content_sha256
- speech_title_from_filename
- forum_scope_inferred
- forum_scope_rule
- event_date
- event_date_source
- language_estimate
- source_url
- source_domain
- source_validation_status
- quality_flags
- text_clean_readable
- text_clean_lower
- tokens_basic
- text_preprocessed_tokens
- text_for_manual_coding
- text_for_bertopic
- char_count_clean
- word_count_clean
- token_count_basic
- unique_token_count_basic
- text_clean_sha256
- preprocessing_config_json
- processed_stage02_at


## 5. Validasi Kolom Wajib

Tahap chunking minimal membutuhkan kolom:

- `speech_id`
- `file_name`
- `text_for_manual_coding`
- `text_for_bertopic`

Kolom metadata lain digunakan bila tersedia. Namun, kolom wajib harus ada agar hasil chunk dapat dilacak kembali ke dokumen asal.

In [5]:
# ============================================================
# Helper Validasi Kolom
# ============================================================

def require_columns(df: pd.DataFrame, required_columns: list[str], df_name: str = "DataFrame") -> None:
    """
    Memastikan DataFrame memiliki kolom yang dibutuhkan.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"{df_name} tidak memiliki kolom wajib: {missing_columns}. "
            f"Kolom tersedia: {list(df.columns)}"
        )


def safe_get(row: pd.Series, column_name: str, default_value=None):
    """
    Mengambil nilai kolom secara aman dari row Pandas.
    """
    if column_name not in row.index:
        return default_value

    value = row[column_name]

    if pd.isna(value):
        return default_value

    return value


REQUIRED_STAGE02_COLUMNS = [
    "speech_id",
    "file_name",
    "text_for_manual_coding",
    "text_for_bertopic"
]

require_columns(speech_df, REQUIRED_STAGE02_COLUMNS, "speech_df")

if speech_df["speech_id"].isna().any():
    raise ValueError("Terdapat speech_id kosong pada speech_df.")

if speech_df["speech_id"].duplicated().any():
    duplicated_ids = speech_df.loc[speech_df["speech_id"].duplicated(), "speech_id"].tolist()
    raise ValueError(f"Terdapat speech_id duplikat: {duplicated_ids}")

if speech_df["text_for_bertopic"].isna().any():
    raise ValueError("Terdapat text_for_bertopic kosong. Periksa output Tahap 02.")

if speech_df["text_for_manual_coding"].isna().any():
    raise ValueError("Terdapat text_for_manual_coding kosong. Periksa output Tahap 02.")

print("Validasi kolom wajib berhasil.")
print(f"Jumlah dokumen pidato: {len(speech_df)}")

Validasi kolom wajib berhasil.
Jumlah dokumen pidato: 6


## 6. Konfigurasi Chunking

Chunking dilakukan dengan pendekatan berbasis kalimat agar konteks teks tetap terbaca.

Konfigurasi utama:

- Target panjang chunk: 220 kata
- Minimal panjang chunk: 80 kata
- Maksimal panjang chunk: 320 kata
- Overlap: 1 kalimat antar chunk

Alasan:

1. Chunk terlalu pendek dapat kehilangan konteks.
2. Chunk terlalu panjang dapat menyulitkan interpretasi Manual Coding.
3. Overlap kecil membantu menjaga kesinambungan konteks antar chunk.
4. Chunk berbasis kalimat lebih aman daripada memotong teks berdasarkan jumlah kata murni.

In [6]:
# ============================================================
# Konfigurasi Chunking
# ============================================================

CHUNKING_CONFIG = {
    "strategy": "sentence_based_chunking_with_small_overlap",
    "target_words": 220,
    "min_words": 80,
    "max_words": 320,
    "overlap_sentences": 1,
    "source_text_for_manual_coding": "text_for_manual_coding",
    "source_text_for_bertopic": "text_for_bertopic",
    "created_at": datetime.now().isoformat(timespec="seconds")
}

print("Konfigurasi chunking:")
print(json.dumps(CHUNKING_CONFIG, indent=2, ensure_ascii=False))

Konfigurasi chunking:
{
  "strategy": "sentence_based_chunking_with_small_overlap",
  "target_words": 220,
  "min_words": 80,
  "max_words": 320,
  "overlap_sentences": 1,
  "source_text_for_manual_coding": "text_for_manual_coding",
  "source_text_for_bertopic": "text_for_bertopic",
  "created_at": "2026-06-11T21:58:28"
}


## 7. Helper Functions untuk Chunking

Cell ini berisi fungsi utama untuk:

1. Normalisasi whitespace.
2. Pemecahan teks menjadi kalimat.
3. Penghitungan kata dan karakter.
4. Pembuatan hash chunk.
5. Pemecahan chunk panjang.
6. Pembuatan quality flags.

In [7]:
# ============================================================
# Helper Functions Chunking
# ============================================================

def normalize_whitespace(text: str) -> str:
    """
    Menormalkan whitespace tanpa mengubah substansi isi teks.
    """
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def count_words(text: str) -> int:
    """
    Menghitung jumlah kata menggunakan regex sederhana.
    """
    if not isinstance(text, str) or not text.strip():
        return 0

    tokens = re.findall(r"\b[\wÀ-ÿ’'-]+\b", text, flags=re.UNICODE)

    return len(tokens)


def count_chars(text: str) -> int:
    """
    Menghitung jumlah karakter.
    """
    if not isinstance(text, str):
        return 0

    return len(text)


def create_text_hash(text: str) -> str:
    """
    Membuat hash SHA-256 untuk teks.
    """
    if not isinstance(text, str):
        text = ""

    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def split_sentences(text: str) -> list[str]:
    """
    Memecah teks menjadi kalimat sederhana.

    Catatan:
    Fungsi ini tidak memakai library NLP eksternal agar notebook mudah direplikasi.
    Pemecahan dilakukan berdasarkan tanda titik, tanda tanya, tanda seru, dan newline.
    """
    text = normalize_whitespace(text)

    if not text:
        return []

    paragraph_candidates = re.split(r"\n+", text)

    sentences = []

    for paragraph in paragraph_candidates:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        parts = re.split(r"(?<=[.!?])\s+", paragraph)

        for part in parts:
            part = part.strip()
            if part:
                sentences.append(part)

    return sentences


def split_long_text_by_words(text: str, max_words: int) -> list[str]:
    """
    Memecah teks yang terlalu panjang berdasarkan jumlah kata.
    Dipakai sebagai fallback jika satu kalimat lebih panjang dari max_words.
    """
    words = re.findall(r"\b[\wÀ-ÿ’'-]+\b|[^\w\s]", text, flags=re.UNICODE)

    if not words:
        return []

    chunks = []
    current = []
    current_word_count = 0

    for token in words:
        current.append(token)

        if re.match(r"\b[\wÀ-ÿ’'-]+\b", token, flags=re.UNICODE):
            current_word_count += 1

        if current_word_count >= max_words:
            chunk_text = " ".join(current)
            chunk_text = re.sub(r"\s+([,.!?;:])", r"\1", chunk_text)
            chunks.append(chunk_text.strip())
            current = []
            current_word_count = 0

    if current:
        chunk_text = " ".join(current)
        chunk_text = re.sub(r"\s+([,.!?;:])", r"\1", chunk_text)
        chunks.append(chunk_text.strip())

    return chunks


def build_chunk_quality_flags(chunk_text: str, min_words: int, max_words: int) -> str:
    """
    Membuat quality flags untuk chunk.
    """
    flags = []

    word_count = count_words(chunk_text)

    if word_count == 0:
        flags.append("EMPTY_CHUNK")

    if word_count < min_words:
        flags.append(f"LOW_WORD_COUNT_LT_{min_words}")

    if word_count > max_words:
        flags.append(f"HIGH_WORD_COUNT_GT_{max_words}")

    if len(chunk_text.strip()) < 50:
        flags.append("VERY_SHORT_TEXT")

    if not flags:
        return "OK"

    return ";".join(flags)


def sentence_based_chunking(
    text: str,
    target_words: int = 220,
    min_words: int = 80,
    max_words: int = 320,
    overlap_sentences: int = 1
) -> list[dict]:
    """
    Membagi teks menjadi chunk berbasis kalimat.

    Prinsip:
    - Kalimat ditambahkan sampai mendekati target_words.
    - Jika chunk melebihi target_words dan sudah memenuhi min_words, chunk disimpan.
    - Overlap kecil dipakai untuk menjaga kesinambungan konteks.
    - Jika satu kalimat terlalu panjang, kalimat tersebut dipecah berdasarkan kata.
    """
    sentences = split_sentences(text)

    if not sentences:
        return []

    normalized_sentences = []
    for sentence in sentences:
        if count_words(sentence) > max_words:
            normalized_sentences.extend(split_long_text_by_words(sentence, max_words=max_words))
        else:
            normalized_sentences.append(sentence)

    chunks = []
    current_sentences = []
    current_word_count = 0
    sentence_start_index = 0

    i = 0
    while i < len(normalized_sentences):
        sentence = normalized_sentences[i]
        sentence_word_count = count_words(sentence)

        if not current_sentences:
            sentence_start_index = i

        current_sentences.append(sentence)
        current_word_count += sentence_word_count

        is_last_sentence = i == len(normalized_sentences) - 1

        should_close_chunk = (
            current_word_count >= target_words
            and current_word_count >= min_words
        ) or is_last_sentence

        if should_close_chunk:
            chunk_text = " ".join(current_sentences).strip()
            chunk_word_count = count_words(chunk_text)

            chunks.append({
                "chunk_text": chunk_text,
                "sentence_start_index": sentence_start_index,
                "sentence_end_index": i,
                "sentence_count": len(current_sentences),
                "chunk_word_count": chunk_word_count,
                "chunk_char_count": count_chars(chunk_text),
                "chunk_hash": create_text_hash(chunk_text),
                "chunk_quality_flags": build_chunk_quality_flags(
                    chunk_text,
                    min_words=min_words,
                    max_words=max_words
                )
            })

            if overlap_sentences > 0 and not is_last_sentence:
                current_sentences = current_sentences[-overlap_sentences:]
                current_word_count = sum(count_words(s) for s in current_sentences)
                sentence_start_index = max(i - overlap_sentences + 1, 0)
            else:
                current_sentences = []
                current_word_count = 0

        i += 1

    return chunks


print("Helper functions untuk chunking berhasil dibuat.")

Helper functions untuk chunking berhasil dibuat.


## 8. Membuat Chunk Master

Pada tahap ini, setiap pidato akan dipecah menjadi beberapa chunk.

Metadata pidato tetap dibawa ke level chunk agar setiap unit analisis dapat ditelusuri kembali ke dokumen asal.

In [8]:
# ============================================================
# Generate Chunk Master
# ============================================================

metadata_columns_optional = [
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "word_count_clean",
    "char_count_clean"
]

chunk_records = []

for _, row in speech_df.iterrows():
    speech_id = row["speech_id"]
    file_name = row["file_name"]

    source_text_column = CHUNKING_CONFIG["source_text_for_bertopic"]
    source_text = row[source_text_column]

    chunks = sentence_based_chunking(
        text=source_text,
        target_words=CHUNKING_CONFIG["target_words"],
        min_words=CHUNKING_CONFIG["min_words"],
        max_words=CHUNKING_CONFIG["max_words"],
        overlap_sentences=CHUNKING_CONFIG["overlap_sentences"]
    )

    if not chunks:
        raise ValueError(f"Tidak ada chunk yang terbentuk untuk speech_id={speech_id}, file_name={file_name}")

    for chunk_index, chunk in enumerate(chunks, start=1):
        chunk_id = f"{speech_id}_CHK_{chunk_index:03d}"

        record = {
            "chunk_id": chunk_id,
            "speech_id": speech_id,
            "file_name": file_name,
            "chunk_order": chunk_index,
            "chunk_text": chunk["chunk_text"],
            "chunk_word_count": chunk["chunk_word_count"],
            "chunk_char_count": chunk["chunk_char_count"],
            "sentence_start_index": chunk["sentence_start_index"],
            "sentence_end_index": chunk["sentence_end_index"],
            "sentence_count": chunk["sentence_count"],
            "chunk_hash": chunk["chunk_hash"],
            "chunk_quality_flags": chunk["chunk_quality_flags"],
            "source_text_column": source_text_column,
            "chunking_strategy": CHUNKING_CONFIG["strategy"],
            "chunking_config_json": json.dumps(CHUNKING_CONFIG, ensure_ascii=False),
            "processed_stage03_at": datetime.now().isoformat(timespec="seconds")
        }

        for col in metadata_columns_optional:
            record[col] = safe_get(row, col, default_value=None)

        chunk_records.append(record)


speech_chunks_master_df = pd.DataFrame(chunk_records)

CHUNK_MASTER_COLUMNS = [
    "chunk_id",
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "source_url",
    "source_domain",
    "source_validation_status",
    "chunk_order",
    "chunk_text",
    "chunk_word_count",
    "chunk_char_count",
    "sentence_start_index",
    "sentence_end_index",
    "sentence_count",
    "chunk_hash",
    "chunk_quality_flags",
    "source_text_column",
    "chunking_strategy",
    "chunking_config_json",
    "word_count_clean",
    "char_count_clean",
    "processed_stage03_at"
]

require_columns(speech_chunks_master_df, CHUNK_MASTER_COLUMNS, "speech_chunks_master_df")
speech_chunks_master_df = speech_chunks_master_df[CHUNK_MASTER_COLUMNS].copy()

display(speech_chunks_master_df.head())
print(f"Jumlah chunk yang terbentuk: {len(speech_chunks_master_df)}")
print(f"Jumlah speech_id unik: {speech_chunks_master_df['speech_id'].nunique()}")

,chunk_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,source_url,source_domain,source_validation_status,...,sentence_end_index,sentence_count,chunk_hash,chunk_quality_flags,source_text_column,chunking_strategy,chunking_config_json,word_count_clean,char_count_clean,processed_stage03_at
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,15,16,e3f58835ac64de59a5c04e132c1f112eaecdfc171fa336...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",254,1642,2026-06-11T21:58:49
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,19,5,768d661bdd3e196267566917363f69a38d216b4d38c046...,LOW_WORD_COUNT_LT_80,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",254,1642,2026-06-11T21:58:49
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,19,20,28d329c7546f5cb19a3aa4329ad7ce62ed7a20ff399728...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",3388,24220,2026-06-11T21:58:49
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,34,16,a210025e9b6d730f70a09850905e7a0098876774ed3b5c...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",3388,24220,2026-06-11T21:58:49
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,https://setkab.go.id/sambutan-presiden-republi...,setkab.go.id,VALID_SETKAB_DOMAIN_OFFLINE,...,49,16,b550fdce49dc7360e42df804c8d8aa5960a4d7f3af7ab2...,OK,text_for_bertopic,sentence_based_chunking_with_small_overlap,"{""strategy"": ""sentence_based_chunking_with_sma...",3388,24220,2026-06-11T21:58:49


Jumlah chunk yang terbentuk: 74
Jumlah speech_id unik: 6


## 9. Validasi Chunk Master

Validasi dilakukan untuk memastikan:

1. `chunk_id` unik.
2. Tidak ada `chunk_text` kosong.
3. Setiap pidato memiliki minimal 1 chunk.
4. Tidak ada duplikasi chunk berdasarkan hash.
5. Jumlah speech yang muncul pada chunk sama dengan jumlah speech pada input.

In [9]:
# ============================================================
# Validasi Chunk Master
# ============================================================

if speech_chunks_master_df["chunk_id"].isna().any():
    raise ValueError("Terdapat chunk_id kosong.")

if speech_chunks_master_df["chunk_id"].duplicated().any():
    duplicated_chunk_ids = speech_chunks_master_df.loc[
        speech_chunks_master_df["chunk_id"].duplicated(),
        "chunk_id"
    ].tolist()
    raise ValueError(f"Terdapat chunk_id duplikat: {duplicated_chunk_ids}")

if speech_chunks_master_df["chunk_text"].isna().any():
    raise ValueError("Terdapat chunk_text kosong.")

if (speech_chunks_master_df["chunk_word_count"] <= 0).any():
    raise ValueError("Terdapat chunk dengan word count tidak valid.")

expected_speech_count = speech_df["speech_id"].nunique()
actual_speech_count = speech_chunks_master_df["speech_id"].nunique()

if actual_speech_count != expected_speech_count:
    raise ValueError(
        f"Jumlah speech_id pada chunk tidak sesuai. "
        f"Expected: {expected_speech_count}, Actual: {actual_speech_count}"
    )

duplicate_chunk_hash_df = speech_chunks_master_df[
    speech_chunks_master_df["chunk_hash"].duplicated(keep=False)
]

if not duplicate_chunk_hash_df.empty:
    print("Peringatan: terdapat potensi duplikasi chunk berdasarkan hash.")
    display(duplicate_chunk_hash_df[["chunk_id", "speech_id", "chunk_hash"]])
else:
    print("Tidak ditemukan duplikasi chunk berdasarkan hash.")

print("Validasi chunk master berhasil.")

Tidak ditemukan duplikasi chunk berdasarkan hash.
Validasi chunk master berhasil.


## 10. Membuat Dataset Khusus BERTopic

Dataset khusus BERTopic dibuat lebih ringkas agar siap digunakan pada tahap modeling.

Kolom utama:

- `doc_id`
- `speech_id`
- `chunk_id`
- `text`
- `forum_scope_inferred`
- `event_date`

Kolom `text` berisi chunk teks yang akan digunakan sebagai dokumen input BERTopic.

In [10]:
# ============================================================
# Dataset Khusus BERTopic
# ============================================================

bertopic_columns = [
    "chunk_id",
    "speech_id",
    "file_name",
    "speech_title_from_filename",
    "forum_scope_inferred",
    "event_date",
    "language_estimate",
    "chunk_order",
    "chunk_text",
    "chunk_word_count",
    "chunk_quality_flags"
]

require_columns(speech_chunks_master_df, bertopic_columns, "speech_chunks_master_df")

speech_chunks_for_bertopic_df = speech_chunks_master_df[bertopic_columns].copy()
speech_chunks_for_bertopic_df = speech_chunks_for_bertopic_df.rename(columns={
    "chunk_id": "doc_id",
    "chunk_text": "text"
})

speech_chunks_for_bertopic_df = speech_chunks_for_bertopic_df[
    [
        "doc_id",
        "speech_id",
        "file_name",
        "speech_title_from_filename",
        "forum_scope_inferred",
        "event_date",
        "language_estimate",
        "chunk_order",
        "text",
        "chunk_word_count",
        "chunk_quality_flags"
    ]
].copy()

display(speech_chunks_for_bertopic_df.head())
print(f"Jumlah dokumen untuk BERTopic: {len(speech_chunks_for_bertopic_df)}")

,doc_id,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_order,text,chunk_word_count,chunk_quality_flags
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,1,Distinguished Leaders of BRICS. It is indeed a...,227,OK
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,"We consider now, this is the time that BRICS m...",42,LOW_WORD_COUNT_LT_80
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,1,Bismillahirrahmanirrahim. Assalamu'alaikum war...,268,OK
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,2,"Yang saya hormati, para Dirut BUMN yang berken...",224,OK
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,3,"Dirut PT Berdikari Saudara Maryadi, Dirut Sang...",242,OK


Jumlah dokumen untuk BERTopic: 74


## 11. Membuat Ringkasan Chunking per Pidato

Ringkasan ini digunakan untuk mengevaluasi distribusi chunk pada setiap pidato.

Informasi yang dihitung:

1. Jumlah chunk per pidato.
2. Total kata hasil chunk.
3. Rata-rata panjang chunk.
4. Minimum dan maksimum panjang chunk.
5. Jumlah chunk yang memiliki quality flag selain `OK`.

In [11]:
# ============================================================
# Chunking Summary by Speech
# ============================================================

chunking_summary_by_speech_df = (
    speech_chunks_master_df
    .groupby(
        [
            "speech_id",
            "file_name",
            "speech_title_from_filename",
            "forum_scope_inferred",
            "event_date",
            "language_estimate"
        ],
        dropna=False
    )
    .agg(
        chunk_count=("chunk_id", "count"),
        total_chunk_words=("chunk_word_count", "sum"),
        avg_chunk_words=("chunk_word_count", "mean"),
        min_chunk_words=("chunk_word_count", "min"),
        max_chunk_words=("chunk_word_count", "max"),
        total_chunk_chars=("chunk_char_count", "sum"),
        flagged_chunk_count=("chunk_quality_flags", lambda x: int((x != "OK").sum()))
    )
    .reset_index()
)

chunking_summary_by_speech_df["avg_chunk_words"] = (
    chunking_summary_by_speech_df["avg_chunk_words"].round(2)
)

display(chunking_summary_by_speech_df)

,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_count,total_chunk_words,avg_chunk_words,min_chunk_words,max_chunk_words,total_chunk_chars,flagged_chunk_count
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,269,134.50,42,227,1704,1
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,17,3859,227.00,127,268,27325,0
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-...,Peresmian Infrastruktur Energi,national,NaN,id,12,2756,229.67,207,247,19383,0
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,Peresmian 166 Sekolah,national,NaN,id,17,3983,234.29,179,348,27964,1
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,World Economic Forum,international,2026-01-22,en,17,3917,230.41,216,263,22928,0
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,Pbb 80,international,NaN,en,9,1961,217.89,119,252,11884,0


## 12. Membuat Quality Report Chunking

Quality report dibuat untuk melihat kelayakan hasil chunking secara keseluruhan.

In [12]:
# ============================================================
# Chunking Quality Report
# ============================================================

quality_records = []

quality_records.append({
    "metric": "input_speech_count",
    "value": int(speech_df["speech_id"].nunique()),
    "description": "Jumlah pidato unik pada input Tahap 02."
})

quality_records.append({
    "metric": "total_chunk_count",
    "value": int(len(speech_chunks_master_df)),
    "description": "Jumlah total chunk yang terbentuk."
})

quality_records.append({
    "metric": "avg_chunk_word_count",
    "value": round(float(speech_chunks_master_df["chunk_word_count"].mean()), 2),
    "description": "Rata-rata jumlah kata per chunk."
})

quality_records.append({
    "metric": "min_chunk_word_count",
    "value": int(speech_chunks_master_df["chunk_word_count"].min()),
    "description": "Jumlah kata terkecil dalam satu chunk."
})

quality_records.append({
    "metric": "max_chunk_word_count",
    "value": int(speech_chunks_master_df["chunk_word_count"].max()),
    "description": "Jumlah kata terbesar dalam satu chunk."
})

quality_records.append({
    "metric": "ok_chunk_count",
    "value": int((speech_chunks_master_df["chunk_quality_flags"] == "OK").sum()),
    "description": "Jumlah chunk dengan quality flag OK."
})

quality_records.append({
    "metric": "flagged_chunk_count",
    "value": int((speech_chunks_master_df["chunk_quality_flags"] != "OK").sum()),
    "description": "Jumlah chunk dengan quality flag selain OK."
})

quality_records.append({
    "metric": "unique_chunk_hash_count",
    "value": int(speech_chunks_master_df["chunk_hash"].nunique()),
    "description": "Jumlah hash chunk unik."
})

quality_records.append({
    "metric": "duplicate_chunk_hash_count",
    "value": int(len(speech_chunks_master_df) - speech_chunks_master_df["chunk_hash"].nunique()),
    "description": "Jumlah chunk yang terindikasi duplikat berdasarkan hash."
})

chunking_quality_report_df = pd.DataFrame(quality_records)

display(chunking_quality_report_df)

print("Distribusi chunk_quality_flags:")
display(
    speech_chunks_master_df["chunk_quality_flags"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={
        "index": "chunk_quality_flags",
        "chunk_quality_flags": "chunk_count"
    })
)

,metric,value,description
0,input_speech_count,6.00,Jumlah pidato unik pada input Tahap 02.
1,total_chunk_count,74.00,Jumlah total chunk yang terbentuk.
2,avg_chunk_word_count,226.28,Rata-rata jumlah kata per chunk.
3,min_chunk_word_count,42.00,Jumlah kata terkecil dalam satu chunk.
4,max_chunk_word_count,348.00,Jumlah kata terbesar dalam satu chunk.
5,ok_chunk_count,72.00,Jumlah chunk dengan quality flag OK.
6,flagged_chunk_count,2.00,Jumlah chunk dengan quality flag selain OK.
7,unique_chunk_hash_count,74.00,Jumlah hash chunk unik.
8,duplicate_chunk_hash_count,0.00,Jumlah chunk yang terindikasi duplikat berdasa...


Distribusi chunk_quality_flags:


,chunk_count,count
0,OK,72
1,LOW_WORD_COUNT_LT_80,1
2,HIGH_WORD_COUNT_GT_320,1


## 13. Validasi Akhir Sebelum Penyimpanan

Sebelum output disimpan, dilakukan validasi akhir:

1. Dataset chunk master tidak kosong.
2. Dataset BERTopic tidak kosong.
3. Summary per speech tidak kosong.
4. Quality report tidak kosong.
5. Kolom wajib output tersedia.

In [13]:
# ============================================================
# Validasi Akhir
# ============================================================

if speech_chunks_master_df.empty:
    raise ValueError("speech_chunks_master_df kosong.")

if speech_chunks_for_bertopic_df.empty:
    raise ValueError("speech_chunks_for_bertopic_df kosong.")

if chunking_summary_by_speech_df.empty:
    raise ValueError("chunking_summary_by_speech_df kosong.")

if chunking_quality_report_df.empty:
    raise ValueError("chunking_quality_report_df kosong.")

require_columns(
    speech_chunks_for_bertopic_df,
    ["doc_id", "speech_id", "text", "chunk_word_count"],
    "speech_chunks_for_bertopic_df"
)

if speech_chunks_for_bertopic_df["doc_id"].duplicated().any():
    raise ValueError("Terdapat doc_id duplikat pada dataset BERTopic.")

if speech_chunks_for_bertopic_df["text"].isna().any():
    raise ValueError("Terdapat text kosong pada dataset BERTopic.")

print("Validasi akhir berhasil. Output siap disimpan.")

Validasi akhir berhasil. Output siap disimpan.


## 14. Menyimpan Output Tahap 03

Output disimpan ke folder:

```text
data/processed/
reports/tables/
```

In [14]:
# ============================================================
# Save Output
# ============================================================

speech_chunks_master_path = PROCESSED_DIR / "speech_chunks_master.csv"
speech_chunks_for_bertopic_path = PROCESSED_DIR / "speech_chunks_for_bertopic.csv"
chunking_summary_by_speech_path = REPORT_TABLE_DIR / "chunking_summary_by_speech.csv"
chunking_quality_report_path = REPORT_TABLE_DIR / "chunking_quality_report.csv"
stage03_output_manifest_path = REPORT_TABLE_DIR / "stage03_output_manifest.json"

speech_chunks_master_df.to_csv(speech_chunks_master_path, index=False, encoding="utf-8-sig")
speech_chunks_for_bertopic_df.to_csv(speech_chunks_for_bertopic_path, index=False, encoding="utf-8-sig")
chunking_summary_by_speech_df.to_csv(chunking_summary_by_speech_path, index=False, encoding="utf-8-sig")
chunking_quality_report_df.to_csv(chunking_quality_report_path, index=False, encoding="utf-8-sig")

stage03_output_manifest = {
    "stage": "03_text_chunking",
    "input_file": str(INPUT_SPEECH_PREPROCESSED),
    "outputs": {
        "speech_chunks_master": str(speech_chunks_master_path),
        "speech_chunks_for_bertopic": str(speech_chunks_for_bertopic_path),
        "chunking_summary_by_speech": str(chunking_summary_by_speech_path),
        "chunking_quality_report": str(chunking_quality_report_path)
    },
    "input_speech_count": int(speech_df["speech_id"].nunique()),
    "total_chunk_count": int(len(speech_chunks_master_df)),
    "chunking_config": CHUNKING_CONFIG,
    "created_at": datetime.now().isoformat(timespec="seconds")
}

stage03_output_manifest_path.write_text(
    json.dumps(stage03_output_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Output Tahap 03 berhasil disimpan:")
print(f"1. {speech_chunks_master_path}")
print(f"2. {speech_chunks_for_bertopic_path}")
print(f"3. {chunking_summary_by_speech_path}")
print(f"4. {chunking_quality_report_path}")
print(f"5. {stage03_output_manifest_path}")

Output Tahap 03 berhasil disimpan:
1. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_chunks_master.csv
2. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\data\processed\speech_chunks_for_bertopic.csv
3. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\chunking_summary_by_speech.csv
4. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\chunking_quality_report.csv
5. D:\DATA-KAMIL\MATKUL\SEMESTER-1\DATA-MINING\topic-modeling-prabowo-un-speech\reports\tables\stage03_output_manifest.json


## 15. Preview Output Akhir

Cell ini menampilkan preview akhir agar hasil chunking dapat dicek secara cepat sebelum digunakan untuk tahap berikutnya.

In [15]:
# ============================================================
# Preview Output Akhir
# ============================================================

preview_columns = [
    "doc_id",
    "speech_id",
    "forum_scope_inferred",
    "chunk_order",
    "chunk_word_count",
    "chunk_quality_flags",
    "text"
]

display(speech_chunks_for_bertopic_df[preview_columns].head(10))

print("Ringkasan chunk per pidato:")
display(chunking_summary_by_speech_df)

print("Quality report:")
display(chunking_quality_report_df)

,doc_id,speech_id,forum_scope_inferred,chunk_order,chunk_word_count,chunk_quality_flags,text
0,SPCH_001_BRICS_LEADERS_CHK_001,SPCH_001_BRICS_LEADERS,international,1,227,OK,Distinguished Leaders of BRICS. It is indeed a...
1,SPCH_001_BRICS_LEADERS_CHK_002,SPCH_001_BRICS_LEADERS,international,2,42,LOW_WORD_COUNT_LT_80,"We consider now, this is the time that BRICS m..."
2,SPCH_002_PANEN_RAYA_CHK_001,SPCH_002_PANEN_RAYA,national,1,268,OK,Bismillahirrahmanirrahim. Assalamu'alaikum war...
3,SPCH_002_PANEN_RAYA_CHK_002,SPCH_002_PANEN_RAYA,national,2,224,OK,"Yang saya hormati, para Dirut BUMN yang berken..."
4,SPCH_002_PANEN_RAYA_CHK_003,SPCH_002_PANEN_RAYA,national,3,242,OK,"Dirut PT Berdikari Saudara Maryadi, Dirut Sang..."
5,SPCH_002_PANEN_RAYA_CHK_004,SPCH_002_PANEN_RAYA,national,4,222,OK,"Walaupun selalu, kita selalu ingat saudara-sau..."
6,SPCH_002_PANEN_RAYA_CHK_005,SPCH_002_PANEN_RAYA,national,5,233,OK,"Dari dulu saya mengerti hal ini, tetapi saya t..."
7,SPCH_002_PANEN_RAYA_CHK_006,SPCH_002_PANEN_RAYA,national,6,234,OK,"Karena itu, saya berjuang terus, saya dituduh ..."
8,SPCH_002_PANEN_RAYA_CHK_007,SPCH_002_PANEN_RAYA,national,7,233,OK,"Dan, saya tidak habis pikir, puluhan tahun par..."
9,SPCH_002_PANEN_RAYA_CHK_008,SPCH_002_PANEN_RAYA,national,8,221,OK,"Politik di Indonesia ini pengorbanan, ingin me..."


Ringkasan chunk per pidato:


,speech_id,file_name,speech_title_from_filename,forum_scope_inferred,event_date,language_estimate,chunk_count,total_chunk_words,avg_chunk_words,min_chunk_words,max_chunk_words,total_chunk_chars,flagged_chunk_count
0,SPCH_001_BRICS_LEADERS,NASKAH-PIDATO-PRABOWO-BRICS-LEADERS.txt,Brics Leaders,international,2025-09-08,en,2,269,134.50,42,227,1704,1
1,SPCH_002_PANEN_RAYA,NASKAH-PIDATO-PRABOWO-PANEN-RAYA.txt,Panen Raya,national,2026-01-07,id,17,3859,227.00,127,268,27325,0
2,SPCH_003_PERESMIAN_INFRASTRUKTUR_ENERGI,NASKAH-PIDATO-PRABOWO-PERESMIAN-INFRASTRUKTUR-...,Peresmian Infrastruktur Energi,national,NaN,id,12,2756,229.67,207,247,19383,0
3,SPCH_004_PERESMIAN_166_SEKOLAH,NASKAH-PIDATO-PRABOWO-PERESMIAN-166-SEKOLAH.txt,Peresmian 166 Sekolah,national,NaN,id,17,3983,234.29,179,348,27964,1
4,SPCH_005_WORLD_ECONOMIC_FORUM,NASKAH-PIDATO-PRABOWO-WORLD-ECONOMIC-FORUM.txt,World Economic Forum,international,2026-01-22,en,17,3917,230.41,216,263,22928,0
5,SPCH_006_PBB_80,NASKAH-PIDATO-PRABOWO-PBB-80.txt,Pbb 80,international,NaN,en,9,1961,217.89,119,252,11884,0


Quality report:


,metric,value,description
0,input_speech_count,6.00,Jumlah pidato unik pada input Tahap 02.
1,total_chunk_count,74.00,Jumlah total chunk yang terbentuk.
2,avg_chunk_word_count,226.28,Rata-rata jumlah kata per chunk.
3,min_chunk_word_count,42.00,Jumlah kata terkecil dalam satu chunk.
4,max_chunk_word_count,348.00,Jumlah kata terbesar dalam satu chunk.
5,ok_chunk_count,72.00,Jumlah chunk dengan quality flag OK.
6,flagged_chunk_count,2.00,Jumlah chunk dengan quality flag selain OK.
7,unique_chunk_hash_count,74.00,Jumlah hash chunk unik.
8,duplicate_chunk_hash_count,0.00,Jumlah chunk yang terindikasi duplikat berdasa...


## 16. Interpretasi Tahap 03

Hasil chunking ini menunjukkan bahwa 6 naskah pidato telah diubah menjadi kumpulan dokumen yang lebih kecil.

Output `speech_chunks_master.csv` digunakan sebagai data master chunk yang lengkap.

Output `speech_chunks_for_bertopic.csv` digunakan sebagai input utama untuk BERTopic pada tahap berikutnya.

Apabila terdapat chunk dengan quality flag selain `OK`, hal tersebut belum tentu salah. Pada pidato yang pendek, chunk dapat memiliki jumlah kata di bawah batas minimal. Kondisi tersebut perlu dicatat sebagai keterbatasan data, bukan langsung dihapus.